# Step 02 — Embedding Extraction

Extracts V-JEPA2 and DINOv2 embeddings using:
- **Top-down sub-view** from the InHARD 3-view composite
- **YOLO person crop** aligned to live inference
- **Temporal attention pooling** (not mean-pool)

Results saved to `outputs/embeddings.npz` (VJEPA) and `outputs/embeddings_dinov2.npz`.

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

In [ ]:
from lib.pipeline import PipelineConfig, step_embeddings
from lib.constants import BACKBONE_VJEPA, BACKBONE_DINOV2

cfg = PipelineConfig(
    clips_per_class           = None,        # ALL clips
    inhard_view               = 'topdown',
    temporal_agg              = 'attention',
    backbones                 = (BACKBONE_VJEPA, BACKBONE_DINOV2),
    skip_embeddings_if_exists = False,       # set True to reuse cached
)

print(f"View: {cfg.inhard_view}  |  temporal_agg: {cfg.temporal_agg}")
print("Starting embedding extraction …")
emb_results = step_embeddings(cfg)
print("Done.")

In [ ]:
# Inspect saved embeddings
from lib.paths import OUTPUTS_DIR

for backbone, info in emb_results.items():
    if info.get('skipped'):
        print(f"{backbone}: skipped (cached)")
        continue
    npz = np.load(info['path'], allow_pickle=True)
    X, y, cls = npz['X'], npz['y'], list(npz['class_names'])
    print(f"\n{backbone}:")
    print(f"  shape     : {X.shape}")
    print(f"  classes   : {len(cls)}")
    from collections import Counter
    counts = Counter(y.tolist())
    for i, name in enumerate(cls):
        bar = '█' * counts.get(i, 0)
        print(f"  {name:30s} {counts.get(i,0):4d} {bar[:50]}")

In [ ]:
# Quick PCA sanity check — are classes separable before training?
from sklearn.decomposition import PCA

for backbone, info in emb_results.items():
    if info.get('skipped'): continue
    npz = np.load(info['path'], allow_pickle=True)
    X, y, cls = npz['X'], npz['y'], list(npz['class_names'])
    X2 = PCA(n_components=2, random_state=42).fit_transform(X)

    fig, ax = plt.subplots(figsize=(9, 7))
    cmap = plt.cm.get_cmap('tab20')
    for i, name in enumerate(cls):
        mask = y == i
        ax.scatter(X2[mask,0], X2[mask,1], c=[cmap(i/max(len(cls)-1,1))],
                   label=name[:16], s=18, alpha=0.55)
    ax.set_title(f'PCA — {backbone} embeddings (before training)', fontsize=12)
    ax.legend(fontsize=7, ncol=2)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / f'02_pca_pretraining_{backbone}.png', dpi=130, bbox_inches='tight')
    plt.show()